In [2]:
import gymnasium as gym
import flappy_bird_gymnasium 
import torch 
import torch.nn as nn 
from collections import deque # for making Replay memory
import random
import itertools # A module for looping indefinetly 
import yaml 
import torch.optim as optim 

In [ ]:
# Does initial device check for better computation
if torch.backend.mps.is_available(): 
    device= "mps"
elif torch.cuda.is_available(): 
    device= "cuda"
else: 
    device= "cpu"

In [ ]:
# creating a replay memory  
class Replaymemory(): 
    # Creating FIFO memory
    def __init__(self, maxlen, seed= None): # maxlen tells how nany experience will memory store, seed makes the random choice to be in same order every-time 
        self.memory= deque([], maxlen=maxlen) # A deque of empyt array with maxlen 

    def append(self, new_exp): 
        self.memory.append(new_exp) # will append experiencec into memory
        
    def sample(self, sample_size):
        return random.sample(self.memory, sample_size) # will return a batch of sample size

    # Private func. to find buffer size
    def __len__(self):
        return len(self.memory)

In [ ]:
# Since policy and terget network has same nn architecture we are making it a class to use it multiple times.
class DQN(nn.Module): 
    # i/p, o/p & hidden dimensions 
    def __init__(self, state_dim=12, action_dim=2, hidden_dim=256): # Given default values
        super(DQN, self).__init__()

        self.model= nn.Sequential(
            nn.Linear(state_dim,hidden_dim), 
            nn.ReLU(), 
            nn.Linear(hidden_dim, action_dim)
        )
    def forward(self, x): 
        return self.model(x)

In [11]:
# A function to test and train 
class agent():
    # importing parametrs 
    
    def __init__(self, parameter_set):
        self.parameter_set= parameter_set
        with open("parameter.yaml", "r") as f: 
            all_param_set= yaml.safe_load(f) 
            params= all_param_set[parameter_set] 

        self.alpha= params["alpha"] 
        self.gamma= params["gamma"]
        self.epsilon_init= params["epsilon_init"] 
        
        self.epsilon_min= params["epsilon_min"] 
        self.episode_decay= params["epsilon_decay"] 
        self.replay_memory_size= params["replay_memory_size"]
        
        self.mini_batch_size= params["mini_batch_size"] 
        self.reward_threshold= params["reward_threshold"] 
        self.network_sync_rate= params["network_sync_rate"]

        #Loss funtion for nn 
        self.loss_fn= nn.MSELoss() 
        self.optimizer= None
    
    def run(self, is_trainig= True, render= False): # Giving default values, so it only renders when its testing
        
        
        env = gym.make("FlappyBird-v0", render_mode="human" if render else None ,use_lidar=True) # rendering only when render = true 
        num_states= env.obervation_space.shape[0] # i/p dimensions
        num_actions= env.action_space.n # o/p dim.
        
        policy_dqn= DQN(num_states, num_actions).to(device) # Created policy dqn and passed it in the found device
       
        if is_trainig: 
            memory = ReplayMemory(self.replay_memory_size)
        
        for episode in itertools.count(): # For all the episodes
            state,_ = env.reset() # resetting env. for each episodes
            episode_reward=0 
            terminated= False 
            
            while not terminated: # For each Episode
                action = env.action_space.sample() # Picking the action
                next_state, reward, terminated,_,_= env.step(action) # Implimenting the action
    
                if is_trainig: 
                    memory.append((state, action,new_state, reward, terminated)) # these are the vales of each new experience, added to replay memory if training.
    
                state= new_state 
                episode_reward+= reward 
            print(f"For episode= {episode+1}, total reward= {episode_reward}")
        # env.close() ,not using this, since we want to stop the code manually